<a href="https://colab.research.google.com/github/pachterlab/cellsweep/blob/main/runtime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# try:
#     from cellsweep import denoise_count_matrix
# except ImportError:
#     print("cellsweep not found, installing...")
#     !pip install -U -q cellsweep[analysis]

In [ ]:
import os
from collections import OrderedDict
import time
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import platform
from cellsweep import denoise_count_matrix
import cellsweep.utils as cs_utils

cellsweep_dir = os.path.dirname(os.path.abspath(""))
rver_docker_workspace = "/home/ruser/work/cellsweep"

# Runtime

pbmc8k dataset: 8k PBMCs from a healthy donor (CellBender Fig2): https://www.10xgenomics.com/datasets/8-k-pbm-cs-from-a-healthy-donor-2-standard-2-1-0

In [ ]:
dataset_name = "pbmc8k"  # options: pbmc8k, hgmm12k, tiny_cellbender, simulation1, custom
verbose = 2  # 2 debug, 1 info, 0 warning, -1 error, -2 critical
scar_env_cpu = "/home/jrich/miniconda3/envs/scar"  # installing scAR: git clone https://github.com/Novartis/scar.git; cd scar; conda env create -f scar-cpu.yml
scar_env_gpu = "/home/jrich/miniconda3/envs/scar_gpu"  # installing scAR: git clone https://github.com/Novartis/scar.git; cd scar; *change name in scar-gpu.yml from scar --> scar_gpu*; conda env create -f scar-gpu.yml
docker = "podman"  # "docker" or "podman" - if podman, then run `sudo setenforce 0` to disable SELinux enforcement before the podman commands

In [ ]:
data_dir = os.path.join(cellsweep_dir, "notebooks", "data", dataset_name, "runtime")
os.makedirs(data_dir, exist_ok=True)

out_dir = os.path.join(cellsweep_dir, "notebooks", "output", dataset_name, "runtime")
os.makedirs(out_dir, exist_ok=True)

if dataset_name == "pbmc8k":
    adata_path_raw = f"{data_dir}/pbmc8k_raw_gene_bc_matrices_h5.h5"
    sequencing_technology = "10XV2"
    model_pkl = "Immune_All_High.pkl"  # path, filename, or url for celltypist model pkl file
    expected_cells = 8381

    if not os.path.exists(adata_path_raw):
        !wget -O {adata_path_raw} https://cf.10xgenomics.com/samples/cell-exp/2.1.0/pbmc8k/pbmc8k_raw_gene_bc_matrices_h5.h5

    matrix_tar_files_dir = os.path.join(data_dir, "matrix_tar_files")
    os.makedirs(matrix_tar_files_dir, exist_ok=True)
    raw_tar_file_dir = os.path.join(matrix_tar_files_dir, "raw_gene_bc_matrices", "GRCh38")
    filtered_tar_file_dir = os.path.join(matrix_tar_files_dir, "filtered_gene_bc_matrices", "GRCh38")
    if not os.path.exists(raw_tar_file_dir):
        raw_tar_path = os.path.join(matrix_tar_files_dir, "pbmc8k_raw_gene_bc_matrices.tar.gz")
        !wget -O {raw_tar_path} https://cf.10xgenomics.com/samples/cell-exp/2.1.0/pbmc8k/pbmc8k_raw_gene_bc_matrices.tar.gz
        !tar -xvzf {raw_tar_path} -C {matrix_tar_files_dir}
    if not os.path.exists(filtered_tar_file_dir):
        filtered_tar_path = os.path.join(matrix_tar_files_dir, "pbmc8k_filtered_gene_bc_matrices.tar.gz")
        !wget -O {filtered_tar_path} https://cf.10xgenomics.com/samples/cell-exp/2.1.0/pbmc8k/pbmc8k_filtered_gene_bc_matrices.tar.gz
        !tar -xvzf {filtered_tar_path} -C {matrix_tar_files_dir}
else:
    raise ValueError(f"Dataset name {dataset_name} not recognized.")

min_genes = 0
min_cells = 0
umi_top_percentile_to_remove = 5
unique_genes_top_percentile_to_remove = 5
mt_gene_percentile_to_remove = 10
max_mt_percentage = None
n_top_genes = 2000
n_pcs = 25
n_neighbors = 20
leiden_resolution = 1.0

try:
    has_gpu = subprocess.call("nvidia-smi", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) == 0
except Exception as e:
    has_gpu = False

max_threads = os.cpu_count()

tool_to_runtime_dict = {}

## System specs

In [ ]:
# CPU and system specs
system = platform.system()
if system == "Darwin":  # MacOS
    !system_profiler SPHardwareDataType
elif system == "Windows":
    !Get-CimInstance Win32_Processor
elif system == "Linux":
    !lscpu
else:
    print(f"Unsupported system: {system}")

# GPU specs
try:
    !nvidia-smi
except Exception as e:
    print("nvidia-smi not found or no GPU available.")


## Raw

In [ ]:
adata_raw = cs_utils.load_adata(adata_path_raw, verbose=verbose)
adata_raw.var_names_make_unique()

adata_raw = cs_utils.infer_empty_droplets(adata_raw, method="threshold", expected_cells=expected_cells, verbose=verbose)  # adds adata.obs["is_empty"]

if "celltype" not in adata_raw.obs.columns:
    adata_raw = cs_utils.determine_cell_types(adata_raw, model_pkl=model_pkl, filter_empty=True, expected_cells=expected_cells, verbose=verbose)

## cellsweep

In [ ]:
threads = 1

if threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")

adata_path_cellsweep = os.path.join(data_dir, f"pbmc8k_cellsweep_{threads}threads.h5ad")
cellsweep_log_file = os.path.join(data_dir, f"pbmc8k_cellsweep_{threads}threads.log")

adata = adata_raw.copy()
if "celltype" not in adata.obs.columns:
    adata = cs_utils.determine_cell_types(adata, model_pkl=model_pkl, filter_empty=True, expected_cells=expected_cells, verbose=verbose)
start_time = time.perf_counter()
_ = denoise_count_matrix(adata, adata_out=adata_path_cellsweep, empty_droplet_method="threshold", expected_cells=expected_cells, threads=threads, verbose=verbose, log_file=cellsweep_log_file)
cellsweep_runtime = time.perf_counter() - start_time
tool_to_runtime_dict[f"cellsweep_{threads}threads"] = cellsweep_runtime

cs_utils.plot_cellsweep_likelihood_over_epochs(log_path=cellsweep_log_file, show=True)

In [ ]:
threads = 16

if threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")
    
adata_path_cellsweep = os.path.join(data_dir, f"cellsweep_{threads}threads.h5ad")
cellsweep_log_file = os.path.join(data_dir, f"cellsweep_{threads}threads.log")

adata = adata_raw.copy()
if "celltype" not in adata.obs.columns:
    adata = cs_utils.determine_cell_types(adata, model_pkl=model_pkl, filter_empty=True, expected_cells=expected_cells, verbose=verbose)
start_time = time.perf_counter()
_ = denoise_count_matrix(adata, adata_out=adata_path_cellsweep, empty_droplet_method="threshold", expected_cells=expected_cells, threads=threads, verbose=verbose, log_file=cellsweep_log_file)
cellsweep_runtime = time.perf_counter() - start_time
tool_to_runtime_dict[f"cellsweep_{threads}threads"] = cellsweep_runtime

cs_utils.plot_cellsweep_likelihood_over_epochs(log_path=cellsweep_log_file, show=True)

In [ ]:
# threads = 64

# if threads > max_threads:
#     SystemExit(f"Requested {threads} threads but only {max_threads} available.")
    
# adata_path_cellsweep = os.path.join(data_dir, f"cellsweep_{threads}threads.h5ad")
# cellsweep_log_file = os.path.join(data_dir, f"cellsweep_{threads}threads.log")

# adata = adata_raw.copy()
# if "celltype" not in adata.obs.columns:
#     adata = cs_utils.determine_cell_types(adata, model_pkl=model_pkl, filter_empty=True, expected_cells=expected_cells, verbose=verbose)
# start_time = time.perf_counter()
# _ = denoise_count_matrix(adata, adata_out=adata_path_cellsweep, empty_droplet_method="threshold", expected_cells=expected_cells, threads=threads, verbose=verbose, log_file=cellsweep_log_file)
# cellsweep_runtime = time.perf_counter() - start_time
# tool_to_runtime_dict[f"cellsweep_{threads}threads"] = cellsweep_runtime

# cs_utils.plot_cellsweep_likelihood_over_epochs(log_path=cellsweep_log_file, show=True)

## CellBender (v0.3.0)

In [ ]:
use_cuda = False
threads = 1

if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")
if not use_cuda and threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")

epochs = 150
runtime = "--cuda" if use_cuda else f"--cpu-threads {threads}"
gpus = "--gpus all" if use_cuda else ""
input_path = adata_path_raw.replace(f"{cellsweep_dir}/notebooks/data", "/data")
output_path = os.path.join(data_dir, f"cellbender_gpu.h5").replace(f"{cellsweep_dir}/notebooks/data", "/data") if use_cuda else os.path.join(data_dir, f"cellbender_cpu_{threads}threads.h5").replace(f"{cellsweep_dir}/notebooks/data", "/data")

start_time = time.perf_counter()
!{docker} run --rm {gpus} -v {cellsweep_dir}/notebooks/data:/data us.gcr.io/broad-dsde-methods/cellbender:0.3.0 \
     cellbender remove-background \
     --input {input_path} \
     --output {output_path} \
     --epochs {epochs} \
     --fpr 0.01 \
     --model full \
     {runtime}
cellbender_runtime = time.perf_counter() - start_time
dict_key = f"cellbender_{'gpu' if use_cuda else f'cpu_{threads}threads'}"
tool_to_runtime_dict[dict_key] = cellbender_runtime

In [ ]:
use_cuda = False
threads = 16

if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")
if not use_cuda and threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")

epochs = 150
runtime = "--cuda" if use_cuda else f"--cpu-threads {threads}"
gpus = "--gpus all" if use_cuda else ""
input_path = adata_path_raw.replace(f"{cellsweep_dir}/notebooks/data", "/data")
output_path = os.path.join(data_dir, f"cellbender_gpu.h5").replace(f"{cellsweep_dir}/notebooks/data", "/data") if use_cuda else os.path.join(data_dir, f"cellbender_cpu_{threads}threads.h5").replace(f"{cellsweep_dir}/notebooks/data", "/data")

start_time = time.perf_counter()
!{docker} run --rm {gpus} -v {cellsweep_dir}/notebooks/data:/data us.gcr.io/broad-dsde-methods/cellbender:0.3.0 \
     cellbender remove-background \
     --input {input_path} \
     --output {output_path} \
     --epochs {epochs} \
     --fpr 0.01 \
     --model full \
     {runtime}
cellbender_runtime = time.perf_counter() - start_time
dict_key = f"cellbender_{'gpu' if use_cuda else f'cpu_{threads}threads'}"
tool_to_runtime_dict[dict_key] = cellbender_runtime

In [ ]:
# use_cuda = False
# threads = 64

# if use_cuda and not has_gpu:
#      SystemExit("CUDA requested but no GPU available.")
# if not use_cuda and threads > max_threads:
#     SystemExit(f"Requested {threads} threads but only {max_threads} available.")

# epochs = 150
# runtime = "--cuda" if use_cuda else f"--cpu-threads {threads}"
# gpus = "--gpus all" if use_cuda else ""
# input_path = adata_path_raw.replace(f"{cellsweep_dir}/notebooks/data", "/data")
# output_path = os.path.join(data_dir, f"cellbender_gpu.h5").replace(f"{cellsweep_dir}/notebooks/data", "/data") if use_cuda else os.path.join(data_dir, f"cellbender_cpu_{threads}threads.h5").replace(f"{cellsweep_dir}/notebooks/data", "/data")

# start_time = time.perf_counter()
# !{docker} run --rm {gpus} -v {cellsweep_dir}/notebooks/data:/data us.gcr.io/broad-dsde-methods/cellbender:0.3.0 \
#      cellbender remove-background \
#      --input {input_path} \
#      --output {output_path} \
#      --epochs {epochs} \
#      --fpr 0.01 \
#      --model full \
#      {runtime}
# cellbender_runtime = time.perf_counter() - start_time
# dict_key = f"cellbender_{'gpu' if use_cuda else f'cpu_{threads}threads'}"
# tool_to_runtime_dict[dict_key] = cellbender_runtime

In [ ]:
use_cuda = True
threads = None

if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")
if not use_cuda and threads > max_threads:
    SystemExit(f"Requested {threads} threads but only {max_threads} available.")

epochs = 150
runtime = "--cuda" if use_cuda else f"--cpu-threads {threads}"
gpus = "--gpus all" if use_cuda else ""
input_path = adata_path_raw.replace(f"{cellsweep_dir}/notebooks/data", "/data")
output_path = os.path.join(data_dir, f"cellbender_gpu.h5").replace(f"{cellsweep_dir}/notebooks/data", "/data") if use_cuda else os.path.join(data_dir, f"cellbender_cpu_{threads}threads.h5").replace(f"{cellsweep_dir}/notebooks/data", "/data")

start_time = time.perf_counter()
!{docker} run --rm {gpus} -v {cellsweep_dir}/notebooks/data:/data us.gcr.io/broad-dsde-methods/cellbender:0.3.0 \
     cellbender remove-background \
     --input {input_path} \
     --output {output_path} \
     --epochs {epochs} \
     --fpr 0.01 \
     --model full \
     {runtime}
cellbender_runtime = time.perf_counter() - start_time
dict_key = f"cellbender_{'gpu' if use_cuda else f'cpu_{threads}threads'}"
tool_to_runtime_dict[dict_key] = cellbender_runtime

## SoupX (v1.6.2)

In [ ]:
adata_soupx_obs_csv = f"{data_dir}/soupx_obs.csv"
if not os.path.exists(adata_soupx_obs_csv):
    adata_soupx_tmp = cs_utils.load_adata(filtered_tar_file_dir)
    adata_soupx_tmp = cs_utils.run_scanpy_preprocessing_and_clustering(adata_soupx_tmp, min_genes=min_genes, min_cells=min_cells, max_mt_percentage=max_mt_percentage, n_top_genes=n_top_genes, n_pcs=n_pcs, n_neighbors=n_neighbors, leiden_resolution=leiden_resolution, seed=42, verbose=verbose)
    adata_soupx_tmp.obs[["leiden"]].to_csv(adata_soupx_obs_csv)

soupx_out_prefix = os.path.join(data_dir, f"soupX")

start_time = time.perf_counter()
!{docker} run --rm \
    -w /home/ruser/work \
    -v {cellsweep_dir}:{rver_docker_workspace} \
    josephrich98/cellsweep_tutorials:soupx.0.1.0 \
    Rscript {rver_docker_workspace}/scripts/run_soupx.R \
        {matrix_tar_files_dir.replace(cellsweep_dir, rver_docker_workspace)} \
        {adata_soupx_obs_csv.replace(cellsweep_dir, rver_docker_workspace)} \
        {soupx_out_prefix.replace(cellsweep_dir, rver_docker_workspace)} \
        leiden
soupx_runtime = time.perf_counter() - start_time
tool_to_runtime_dict["soupx"] = soupx_runtime

## DecontX (v1.8.0)

In [ ]:
decontx_out_prefix = os.path.join(data_dir, f"decontX")

start_time = time.perf_counter()
!{docker} run --rm \
    -w /home/ruser/work \
    -v {cellsweep_dir}:{rver_docker_workspace} \
    josephrich98/cellsweep_tutorials:decontx.0.1.0 \
    Rscript {rver_docker_workspace}/scripts/run_decontx.R \
        {raw_tar_file_dir.replace(cellsweep_dir, rver_docker_workspace)} \
        {filtered_tar_file_dir.replace(cellsweep_dir, rver_docker_workspace)} \
        {sequencing_technology} \
        {decontx_out_prefix.replace(cellsweep_dir, rver_docker_workspace)} \
        --dont_prepend_sample_to_barcodes
decontx_runtime = time.perf_counter() - start_time
tool_to_runtime_dict["decontx"] = decontx_runtime

## scAR (v0.7.0)

In [ ]:
%env MPLBACKEND=
use_cuda = False

if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")

epochs = 150
adata_path_scar = os.path.join(data_dir, f"scar_gpu.h5ad") if use_cuda else os.path.join(data_dir, f"scar_cpu_{threads}threads.h5ad")

runtime = "--cuda" if use_cuda else ""
conda_run_flag = "-p" if "/" in scar_env_cpu else "-n"
start_time = time.perf_counter()
!conda run {conda_run_flag} {scar_env_cpu} \
     python {cellsweep_dir}/scripts/run_scar.py \
     -r {raw_tar_file_dir} \
     -f {filtered_tar_file_dir} \
     -o {adata_path_scar} \
     {runtime} \
     --epochs {epochs}
scar_runtime = time.perf_counter() - start_time
dict_key = f"scar_{'gpu' if use_cuda else 'cpu'}"
tool_to_runtime_dict[dict_key] = scar_runtime

In [ ]:
%env MPLBACKEND=
use_cuda = True

if use_cuda and not has_gpu:
     SystemExit("CUDA requested but no GPU available.")

epochs = 150
adata_path_scar = os.path.join(data_dir, f"scar_gpu.h5ad") if use_cuda else os.path.join(data_dir, f"scar_cpu_{threads}threads.h5ad")

runtime = "--cuda" if use_cuda else ""
conda_run_flag = "-p" if "/" in scar_env_gpu else "-n"
start_time = time.perf_counter()
!conda run {conda_run_flag} {scar_env_gpu} \
     python {cellsweep_dir}/scripts/run_scar.py \
     -r {raw_tar_file_dir} \
     -f {filtered_tar_file_dir} \
     -o {adata_path_scar} \
     {runtime} \
     --epochs {epochs}
scar_runtime = time.perf_counter() - start_time
dict_key = f"scar_{'gpu' if use_cuda else 'cpu'}"
tool_to_runtime_dict[dict_key] = scar_runtime

In [ ]:
# print(tool_to_runtime_dict)

In [ ]:
# convert to minutes
# tool_to_runtime_minutes_dict = {'cellsweep_1threads': 4.50172119550407, 'cellsweep_16threads': 0.44180107543555397, 'cellbender_cpu_1threads': 504.35386805809105, 'cellbender_cpu_16threads': 128.92702953227175, 'cellbender_gpu': 22.02172445510514, 'soupx': 1.6098512041382491, 'decontx': 2.1877690135035666, 'scar_cpu': 24.54572350685485, 'scar_gpu': 13.524172533024103}
tool_to_runtime_minutes_dict = {k: v / 60 for k, v in tool_to_runtime_dict.items()}

import re
def thread_count(tool_name):
    m = re.search(r"_(\d+)threads$", tool_name)
    return int(m.group(1)) if m else float("inf")

# reorder - cellsweep first, then alphabetical
tool_to_runtime_minutes_dict = dict(
    sorted(
        tool_to_runtime_minutes_dict.items(),
        key=lambda kv: (
            not kv[0].startswith("cellsweep"),  # False (0) first
            thread_count(kv[0]),                # fewer threads first
            kv[0],                              # alphabetical
        )
    )
)

print(tool_to_runtime_minutes_dict)

In [ ]:
cs_utils.plot_runtime_comparison(tool_to_runtime_minutes_dict, plot_log_inset=False, inset_limit=50, out_path=os.path.join(out_dir, "runtime_comparison.png"))
cs_utils.plot_runtime_comparison(tool_to_runtime_minutes_dict, log=True, out_path=os.path.join(out_dir, "runtime_comparison_log.png"))

bar_color_prefix_color_map = {
    "cellsweep": '#1f77b4',
    "cellbender": '#ff7f0e',
    "decontx": '#2ca02c',
    "scar": '#d62728',
    "soupx": '#9467bd',
}

# bar_color_prefix_color_map = {
#     "cellsweep": '#9467bd',
#     "cellbender": '#d62728',
#     "decontx": '#2ca02c',
#     "scar": '#ff7f0e',
#     "soupx": '#1f77b4',
# }
# bar_color_prefix_color_map = True
cs_utils.plot_runtime_comparison(tool_to_runtime_minutes_dict, order="ascending", bar_color_prefix_color_map=bar_color_prefix_color_map, plot_log_inset=True, inset_limit=30, out_path=os.path.join(out_dir, "runtime_ascending_comparison.png"))
cs_utils.plot_runtime_comparison(tool_to_runtime_minutes_dict, order="ascending", bar_color_prefix_color_map=bar_color_prefix_color_map, log=True, out_path=os.path.join(out_dir, "runtime_ascending_comparison_log.png"))

cs_utils.plot_legend_only(
    bar_color_prefix_color_map,
    out_path=os.path.join(out_dir, "runtime_legend.png"),
    title="Tool",
)

## no empties

In [ ]:
# tool_to_runtime_no_empties_dict = {}

In [ ]:
# threads = 16

# if threads > max_threads:
#     SystemExit(f"Requested {threads} threads but only {max_threads} available.")
    
# adata_path_cellsweep = os.path.join(data_dir, f"cellsweep_{threads}threads_no_empties.h5ad")
# cellsweep_log_file = os.path.join(data_dir, f"cellsweep_{threads}threads_no_empties.log")

# adata = adata_raw.copy()
# if "celltype" not in adata.obs.columns:
#     adata = cs_utils.determine_cell_types(adata, model_pkl=model_pkl, filter_empty=True, expected_cells=expected_cells, verbose=verbose)
# adata = adata[~adata.obs["is_empty"]].copy()  # new
# start_time = time.perf_counter()
# _ = denoise_count_matrix(adata, adata_out=adata_path_cellsweep, beta=0.03, eps=1e-9, empty_droplet_method="threshold", expected_cells=expected_cells, threads=threads, verbose=verbose, log_file=cellsweep_log_file)
# cellsweep_runtime = time.perf_counter() - start_time
# tool_to_runtime_no_empties_dict[f"cellsweep_{threads}threads_no_empties"] = cellsweep_runtime
# tool_to_runtime_no_empties_dict[f"cellsweep_{threads}threads"] = tool_to_runtime_dict[f"cellsweep_{threads}threads"]

# cs_utils.plot_cellsweep_likelihood_over_epochs(log_path=cellsweep_log_file, show=True)

In [ ]:
# tool_to_runtime_no_empties_minutes_dict = {k: v / 60 for k, v in tool_to_runtime_no_empties_dict.items()}

In [ ]:
# bar_color_prefix_color_map = {
#     "cellsweep": '#1f77b4',
#     "cellsweep_no_empties": '#ff7f0e',
# }

# cs_utils.plot_runtime_comparison(tool_to_runtime_no_empties_minutes_dict, order="ascending", bar_color_prefix_color_map=bar_color_prefix_color_map, plot_log_inset=False, inset_limit=30, out_path=os.path.join(out_dir, "runtime_no_empties_ascending_comparison.png"))

# cs_utils.plot_legend_only(
#     bar_color_prefix_color_map,
#     out_path=os.path.join(out_dir, "runtime_legend.png"),
#     title="Tool",
# )